# 7. 객체 탐지(Object Detection)와 YOLO

> 이 노트북은 강의 자료에서 자동 변환되었습니다.  
> Google Colab에서 `런타임 > 런타임 유형 변경 > GPU`로 설정 후 실행하세요.


# 7. 객체 탐지(Object Detection)와 YOLO

## 학습 목표

1. 객체 탐지의 개념과 Bounding Box, IOU를 설명할 수 있다
2. NMS의 동작 과정과 필요성을 이해할 수 있다
3. 정밀도, 재현율, AP, mAP 지표를 해석할 수 있다
4. 객체 탐지 모델의 발전 흐름(RCNN → YOLO)을 설명할 수 있다
5. Ultralytics YOLO를 사용하여 커스텀 데이터셋으로 모델을 학습할 수 있다
6. 웹캠을 활용한 실시간 객체 탐지를 구현할 수 있다

## 진행 순서

1. [객체 탐지 개요](#part1)
2. [NMS와 모델 성능 평가](#part2)
3. [데이터셋](#part3)
4. [객체 탐지의 역사](#part4)
5. [YOLO 실습: Ultralytics](#part5)
6. [YOLO 실습: 웹캠 객체 탐지](#part6)
7. [통합 정리](#part7)

---

## 1. 객체 탐지 개요

**학습목표**: 객체 탐지의 개념과 Bounding Box, IOU를 설명할 수 있다

- 한 이미지에서 객체와 그 경계 상자(bounding box)를 탐지
- 객체 탐지 알고리즘은 일반적으로 이미지를 입력으로 받고, 경계 상자와 객체 클래스 리스트를 출력
- 경계 상자에 대해 그에 대응하는 예측 클래스와 클래스의 신뢰도(confidence)를 출력

### 활용분야

- 자율 주행 자동차에서 다른 자동차와 보행자를 찾을 때
- 의료 분야에서 방사선 사진을 사용해 종양이나 위험한 조직을 찾을 때
- 제조업에서 조립 로봇이 제품을 조립하거나 수리할 때
- 보안 산업에서 위협을 탐지하거나 사람 수를 셀 때

### Bounding Box

- 이미지에서 하나의 객체 전체를 포함하는 가장 작은 직사각형

![](./img/yolo/object_detaction001.png)

### IOU(Intersection Over Union)

- 실측값(Ground Truth)과 모델이 예측한 값이 얼마나 겹치는지를 나타내는 지표
  ![](./img/yolo/object_detaction002.png)

- IOU가 높을수록 잘 예측한 모델
  ![](./img/yolo/object_detaction003.png)

<br>

- 예시

  ![](./img/yolo/object_detaction004.png)

**핵심**: 객체 탐지는 이미지에서 객체의 위치(Bounding Box)와 클래스를 동시에 예측한다. IOU는 예측과 실측의 겹침 비율로, 탐지 품질을 측정하는 핵심 지표이다.

---

## 2. NMS와 모델 성능 평가

**학습목표**: NMS의 동작 과정과 필요성을 이해하고, 정밀도, 재현율, AP, mAP 지표를 해석할 수 있다

### NMS(Non-Maximum Suppression, 비최댓값 억제)

- 확률이 가장 높은 상자와 겹치는 상자들을 제거하는 과정
- 최댓값을 갖지 않는 상자들을 제거
- 과정
  1. 확률 기준으로 모든 상자를 정렬하고 먼저 가장 확률이 높은 상자를 취함
  2. 각 상자에 대해 다른 모든 상자와의 IOU를 계산
  3. 특정 임곗값을 넘는 상자는 제거

  ![](./img/yolo/object_detaction005.png)

### 모델 성능 평가

#### 정밀도(Precision)와 재현율(Recall)

- 일반적으로 객체 탐지 모델 평가에 사용되지는 않지만, 다른 지표를 계산하는 기본 지표 역할을 함

  - True Positives(`TP`): 예측이 동일 클래스의 실제 상자와 일치하는지 측정
  - False Positives(`FP`): 예측이 실제 상자와 일치하지 않는지 측정
  - False Negatives(`FN`): 실제 분류값이 그와 일치하는 예측을 갖지 못하는지 측정

  \[
  precision = \frac{TP}{TP + FP}
  \]
  \[
  recall = \frac{TP}{TP + FN}
  \]

  - 모델이 안정적이지 않은 특징을 기반으로 객체 존재를 예측하면 거짓긍정(FP)이 많아져서 정밀도가 낮아짐
  - 모델이 너무 엄격해서 정확한 조건을 만족할 때만 객체가 탐지된 것으로 간주하면 거짓부정(FN)이 많아져서 재현율이 낮아짐

#### 정밀도-재현율 곡선(Precision-Recall Curve)

- 신뢰도 임계값마다 모델의 정밀도와 재현율을 시각화
- 모든 bounding box와 함께 모델이 예측의 정확성을 얼마나 확실하는지 0 ~ 1사이의 숫자로 나타내는 신뢰도를 출력
- 임계값 T에 따라 정밀도와 재현율이 달라짐
  - 임계값 T 이하의 예측은 제거함
  - T가 1에 가까우면 정밀도는 높지만 재현율은 낮음
  - 놓치는 객체가 많아져서 재현율이 낮아짐. 즉, 신뢰도가 높은 예측만 유지하기때문에 정밀도는 높아짐
  - T가 0에 가까우면 정밀도는 낮지만 재현율은 높음
  - 대부분의 예측을 유지하기때문에 재현율은 높아지고, 거짓긍정(FP)이 많아져서 정밀도가 낮아짐
- 예를 들어, 모델이 보행자를 탐지하고 있으면 특별한 이유없이 차를 세우더라도 어떤 보행자도 놓치지 않도록 재현율을 높여야 함
- 모델이 투자 기회를 탐지하고 있다면 일부 기회를 놓치게 되더라도 잘못된 기회에 돈을 거는 일을 피하기 위해 정밀도를 높여야 함

![](./img/yolo/object_detaction006.png)

#### AP (Average Precision, 평균 정밀도) 와 mAP(mean Average Precision)

- 곡선의 아래 영역에 해당
- 항상 1x1 정사각형으로 구성되어 있음
  즉, 항상 0 ~ 1 사이의 값을 가짐
- 단일 클래스에 대한 모델 성능 정보를 제공
- 전역 점수를 얻기위해서 mAP를 사용
- 예를 들어, 데이터셋이 10개의 클래스로 구성된다면 각 클래스에 대한 AP를 계산하고, 그 숫자들의 평균을 다시 구함

- mAP 사용
  - 최소 2개 이상의 객체를 탐지하는 대회인 PASCAL Visual Object Classes와 Common Objects in Context(COCO)에서 mAP가 사용됨
  - COCO 데이터셋이 더 많은 클래스를 포함하고 있기 때문에 보통 Pascal VOC보다 점수가 더 낮게 나옴

    ![](./img/yolo/object_detaction007.png)

**핵심**: NMS는 중복된 예측 상자를 제거하여 최종 탐지 결과를 정리한다. mAP는 모든 클래스의 AP 평균으로, 객체 탐지 모델의 종합 성능을 나타낸다.

---

## 3. 데이터셋

**학습목표**: 객체 탐지의 대표적인 벤치마크 데이터셋을 이해할 수 있다

### VOC Dataset

- 2005년부터 2012년까지 진행
- Object Detection 기술의 benchmark로 간주
- 데이터셋에는 20개의 클래스가 존재
- 훈련 및 검증 데이터 : 11,530개
- ROI에 대한 27,450개의 Annotation이 존재
- 이미지당 2.4개의 객체 존재

  ![](./img/yolo/object_detaction008.png)


### COCO Dataset

- Common Objects in Context
- 200,000개의 이미지
- 80개의 카테고리에 500,000개 이상의 객체 Annotation이 존재

- https://cocodataset.org/

![](./img/yolo/object_detaction009.png)

**핵심**: VOC(20 클래스)와 COCO(80 클래스)는 객체 탐지의 대표적인 벤치마크 데이터셋이다.

---

## 4. 객체 탐지의 역사

**학습목표**: 객체 탐지 모델의 발전 흐름(RCNN → YOLO)을 설명할 수 있다

![](./img/yolo/object_detaction010.png)

* RCNN (2013)
  - Rich feature hierarchies for accurate object detection and semantic segmentation (https://arxiv.org/abs/1311.2524)
  - 물체 검출에 사용된 기존 방식인 sliding window는 background를 검출하는 소요되는 시간이 많았는데, 이를 개선시킨 기법으로 Region Proposal 방식 제안
  - 매우 높은 Detection이 가능하지만, 복잡한 아키텍처 및 학습 프로세스로 인해 Detection 시간이 매우 오래 걸림

* SPP Net (2014)
  - Spatial Pyramid Pooling in Deep Convolutional Networks for Visual Recognition (https://arxiv.org/abs/1406.4729)
  - RCNN의 문제를 Selective search로 해결하려 했지만, bounding box의 크기가 제각각인 문제가 있어서 FC Input에 고정된 사이즈로 제공하기 위한 방법 제안
  - SPP은 RCNN에서 conv layer와 fc layer사이에 위치하여 서로 다른 feature map에 투영된 이미지를 고정된 값으로 풀링
  - SPP를 이용해 RCNN에 비해 실행시간을 매우 단축시킴

* Fast RCNN (2015)
  - Fast R-CNN (https://arxiv.org/abs/1504.08083)
  - SPP layer를 ROI pooling으로 바꿔서 7x7 layer 1개로 해결
  - SVM을 softmax로 대체하여 Classification 과 Regression Loss를 함께 반영한 Multi task Loss 사용
  - ROI Pooling을 이용해 SPP보다 간단하고, RCNN에 비해 수행시간을 많이 줄임

* Fater RCNN(2015)
  - Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks (https://arxiv.org/abs/1506.01497)
  - RPN(Region proposal network) + Fast RCNN 방식
  - Selective Search를 대체하기 위한 Region Proposal Network구현
  - RPN도 학습시켜서 전체를 end-to-end로 학습 가능 (GPU사용 가능)
  - Region Proposal를 위해 Object가 있는지 없는지의 후보 Box인 Anchor Box 개념 사용
  - Anchor Box를 도입해 FastRCNN에 비해 정확도를 높이고 속도를 향상시킴

* SSD (2015)
  - SSD: Single Shot MultiBox Detector (https://arxiv.org/abs/1512.02325)
  - Faster-RCNN은 region proposal과 anchor box를 이용한 검출의 2단계를 걸치는 과정에서 시간이 필요해 real-time(20~30 fps)으로는 어려움
  - SSD는 Feature map의 size를 조정하고, 동시에 앵커박스를 같이 적용함으로써 1 shot으로 물체 검출이 가능
  - real-time으로 사용할 정도의 성능을 갖춤 (30~40 fps)
  - 작은 이미지의 경우에 잘 인식하지 못하는 경우가 생겨서 data augmentation을 통해 mAP를 63에서 74로 비약적으로 높임

* RetinaNet (2017)
  - Focal Loss for Dense Object Detection (https://arxiv.org/abs/1708.02002)
  - RetinaNet이전에는 1-shot detection과 2-shot detection의 차이가 극명하게 나뉘어 속도를 선택하면 정확도를 trade-off 할 수 밖에 없는 상황
  - RetinaNet은 Focal Loss라는 개념의 도입과 FPN 덕분에 기존 모델들보다 정확도도 높고 속도도 여타 1-shot detector와 비견되는 모델
  - Detection에선 검출하고 싶은 물체와 (foreground object) 검출할 필요가 없는 배경 물체들이 있는데 (background object) 배경 물체의 숫자가 매우 많을 경우 배경 Loss를 적게 하더라도 숫자에 압도되어 배경의 Loss의 총합을 학습해버림 (예를 들어, 숲을 배경으로 하는 사람을 검출해야하는데 배경의 나무가 100개나 되다보니 사람의 특징이 아닌 나무가 있는 배경을 학습해버림)
  - Focal Loss는 이런 문제를 기존의 crossentropy 함수에서 (1-sig)을 제곱하여 background object의 loss를 현저히 줄여버리는 방법으로 loss를 변동시켜 해결
  - Focal Loss를 통해 검출하고자 하는 물체와 관련이 없는 background object들은 학습에 영향을 주지 않게 되고, 학습의 다양성이 더 넓어짐 (작은 물체, 큰 물체에 구애받지 않고 검출할 수 있게됨)
  - 실제로 RetinaNet은 object proposal을 2000개나 실시하여 이를 확인

* Mask R-CNN (2018)
  - Mask R-CNN (https://arxiv.org/pdf/1703.06870.pdf)

* YOLO (2018)
  - YOLOv3: An Incremental Improvement (https://arxiv.org/abs/1804.02767)
  - YOLO는 v1, v2, v3의 순서로 발전하였는데, v1은 정확도가 너무 낮은 문제가 있었고 이 문제는 v2까지 이어짐
  - 엔지니어링적으로 보완한 v3는 v2보다 살짝 속도는 떨어지더라도 정확도를 대폭 높인 모델
  - RetinaNet과 마찬가지로 FPN을 도입해 정확도를 높임
  - RetinaNet에 비하면 정확도는 4mAP정도 떨어지지만, 속도는 더 빠르다는 장점

* RefineDet (2018)
  - Single-Shot Refinement Neural Network for Object Detection (https://arxiv.org/pdf/1711.06897.pdf)

* M2Det (2019)
  - M2Det: A Single-Shot Object Detector based on Multi-Level Feature Pyramid Network (https://arxiv.org/pdf/1811.04533.pdf)

* EfficientDet (2019)
  - EfficientDet: Scalable and Efficient Object Detection (https://arxiv.org/pdf/1911.09070v1.pdf)

* YOLOv4 (2020)
  - YOLOv4: Optimal Speed and Accuracy of Object Detection (https://arxiv.org/pdf/2004.10934v1.pdf)
  - YOLOv3에 비해 AP, FPS가 각각 10%, 12% 증가
  - YOLOv3와 다른 개발자인 AlexeyBochkousky가 발표
  - v3에서 다양한 딥러닝 기법(WRC, CSP ...) 등을 사용해 성능을 향상시킴
  - CSPNet 기반의 backbone(CSPDarkNet53)을 설계하여 사용

* YOLOv5 (2020)
  - YOLOv4에 비해 낮은 용량과 빠른 속도 (성능은 비슷)
  - YOLOv4와 같은 CSPNet 기반의 backbone을 설계하여 사용
  - YOLOv3를 PyTorch로 implementation한 GlennJocher가 발표
  - Darknet이 아닌 PyTorch 구현이기 때문에, 이전 버전들과 다르다고 할 수 있음

* YOLOv6 (2022)

	•	BYD에 의해 발표된 YOLO 버전으로, 산업용 컴퓨터 비전 작업에 최적화된 모델.
	•	Anchor-Free 구조를 도입하여 더 빠르고 정확한 탐지가 가능.
	•	Nano, Tiny, Small, Medium 모델 크기 제공으로 다양한 컴퓨팅 환경에 대응 가능.
	•	MobileNet 기반의 백본을 사용하여 경량화 및 효율성 향상.
	•	TensorRT 및 ONNX 지원으로, 엣지 장치와 임베디드 시스템에서 효율적인 배포 가능.
	•	YOLOv5와 비교했을 때, 작은 객체 탐지에 더 우수한 성능을 보임.
	•	고정밀한 산업 애플리케이션에서의 속도와 정확도를 동시에 달성.

* YOLOv7 (2022)

	•	Wang, Bochkovskiy, Liao에 의해 발표된 YOLO 버전으로, 속도와 정확도 측면에서 가장 우수한 성능을 보여주는 모델.
	•	계층적 가중치 학습(Extended-ELAN)을 통해 효율적인 학습 및 성능 향상.
	•	다중 스케일 학습(Multiple Scale Learning) 기능으로 작은 객체 탐지 능력이 크게 향상됨.
	•	YOLO 시리즈 중 가장 높은 속도-정확도 균형을 자랑하며, SOTA (State-of-the-Art) 성능 달성.
	•	YOLOv5보다 더 작은 모델 크기와 더 빠른 속도를 제공하며, 다양한 크기 모델 제공 (YOLOv7-tiny, YOLOv7-large).
	•	기존 YOLO 모델에 비해 레이어 간 연결이 개선되어 네트워크 효율성이 증가.
	•	객체 탐지, 이미지 분류, 이미지 분할 등 다양한 작업에서 뛰어난 성능을 보임.

* YOLOv8 (2023)

	•	이전 버전들과 비교했을 때 더 가볍고 더 나은 정확도 제공.
	•	모듈화된 디자인: 다양한 작업에 쉽게 적용할 수 있도록 설계됨. 객체 탐지, 분할, 분류 등을 단일 프레임워크에서 지원.
	•	다중 태스크 지원: 객체 탐지뿐 아니라, **이미지 분할(Image Segmentation)**과 이미지 분류(Image Classification) 작업도 지원.
	•	자동 하이퍼파라미터 튜닝: 학습 중 최적의 하이퍼파라미터를 자동으로 설정해 성능 극대화.
	•	다양한 형식으로 모델 배포 가능: PyTorch, TensorRT, ONNX, CoreML 등 여러 플랫폼에서 실행 가능.
	•	백본 아키텍처가 최적화되어, 모바일 기기와 임베디드 장치에서도 빠르고 효율적으로 동작.
	•	사용자 친화적인 API 제공으로, 개발자들이 쉽게 모델을 배포하고 응용할 수 있음.

**핵심**: 객체 탐지는 2-stage(RCNN 계열)에서 1-stage(SSD, YOLO)로 발전하며 속도와 정확도의 균형을 개선해 왔다.

---

## 5. YOLO 실습: Ultralytics

**학습목표**: Ultralytics YOLO를 사용하여 커스텀 데이터셋으로 모델을 학습할 수 있다

### Ultralytics

Ultralytics는 YOLOv5부터 YOLO 모델의 개발과 지원을 시작.
Ultralytics YOLO 모델을 사용하여 다양한 작업(객체 탐지, 분류, 분할,)을 수행할 수 있습니다. YOLO 모델의 각 모드는 주로 4가지로 구분되며, 각 모드는 목적에 따라 다른 방식으로 모델을 사용할 수 있습니다.


In [ ]:
!pip install ultralytics

In [ ]:
from IPython.display import Image
Image(filename='bus.jpg',width=600)

In [ ]:
Image(filename='runs/detect/predict2/bus.jpg',width=600)

### 포트홀 탐지 모델

데이터셋 다운로드
포트홀 데이터셋: [https://public.roboflow.com/object-detection/pothole](https://public.roboflow.com/object-detection/pothole)


In [ ]:
%mkdir -p /content/datasets/pothole
%cd /content/datasets/pothole

In [ ]:
from glob import glob

train_img_list = glob('/content/datasets/pothole/train/images/*.jpg')
test_img_list = glob('/content/datasets/pothole/test/images/*.jpg')
valid_img_list = glob('/content/datasets/pothole/valid/images/*.jpg')
print('train:',len(train_img_list),'valid:',len(valid_img_list),'test:',len(test_img_list))

In [ ]:
%cd /content/
%pwd

In [ ]:
%ls runs/detect/train

In [ ]:
Image(filename='runs/detect/train/val_batch2_labels.jpg',width=1000)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**핵심**: Ultralytics YOLO는 간단한 명령으로 학습, 검증, 예측을 수행할 수 있다. 커스텀 데이터셋(포트홀 등)으로 Fine-tuning이 가능하다.

---

## 6. YOLO 실습: 웹캠 객체 탐지

**학습목표**: 웹캠을 활용한 실시간 객체 탐지를 구현할 수 있다

### Colab에서 웹캠 사용하여 객체 탐지


In [ ]:
from ultralytics import YOLO
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
from PIL import Image
from IPython.display import display

# 웹캠 이미지를 캡처하는 JavaScript 코드
def capture_image():
    js = Javascript('''
    async function captureImage() {
        const video = document.createElement('video');
        const stream = await navigator.mediaDevices.getUserMedia({ video: true });

        video.srcObject = stream;
        await new Promise((resolve) => {
            video.onloadedmetadata = () => {
                resolve();
            };
        });
        video.play();

        // 지연 시간 추가 (예: 1초)
        await new Promise((resolve) => setTimeout(resolve, 1000));

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        const context = canvas.getContext('2d');
        context.drawImage(video, 0, 0, canvas.width, canvas.height);

        stream.getVideoTracks()[0].stop();

        return canvas.toDataURL('image/jpeg', 0.8);
    }
    captureImage();
    ''')
    display(js)
    data = eval_js('captureImage()')
    return data

# base64 데이터를 OpenCV 이미지로 변환
def decode_image(data):
    img_str = b64decode(data.split(',')[1])
    np_arr = np.frombuffer(img_str, np.uint8)
    img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return img

# YOLO 모델 로드
model = YOLO("yolo11n.pt")

# 웹캠에서 이미지 캡처 및 디코딩
data = capture_image()
img = decode_image(data)
results = model(img)

# 첫 번째 탐지 결과 선택
result = results[0]  # 첫 번째 결과 선택

# 탐지된 객체 확인
if result.boxes:  # 탐지된 객체가 있는지 확인
    print("탐지된 객체가 있습니다.")

    # boxes에서 좌표 및 클래스 ID 가져오기
    boxes = result.boxes.xyxy.cpu().numpy()  # 바운딩 박스 좌표
    confs = result.boxes.conf.cpu().numpy()  # 신뢰도
    class_ids = result.boxes.cls.cpu().numpy().astype(int)  # 클래스 ID

    # 클래스 이름 가져오기
    names = result.names

    # 바운딩 박스 그리기
    for box, conf, class_id in zip(boxes, confs, class_ids):
        x_min, y_min, x_max, y_max = map(int, box)  # 바운딩 박스 좌표
        label = f"{names[class_id]} {conf:.2f}"  # 클래스 이름 및 신뢰도
        color = (0, 255, 0)  # 녹색 (BGR 형식)

        # 바운딩 박스 그리기 (OpenCV)
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color, 2)

        # 라벨 표시
        cv2.putText(img, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    # 바운딩 박스가 그려진 이미지를 PIL 이미지로 변환
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    # Colab에서 이미지를 출력
    display(img_pil)

else:
    print("탐지된 객체가 없습니다.")

### YOLO 실시간 웹캠 탐지

YOLO 개체 탐지 모델을 사용하여 실시간 웹캠을 통해 객체를 감지하고 감지된 객체의 경계 상자와 라벨을 화면에 표시


In [ ]:
from ultralytics import YOLO

# Load your model

model = YOLO('yolo11n.pt')
results= model.predict(source=0,show=True,show_labels=True,save=True,stream=True)
# 저장된 경로 확인
for r in results:
    boxes = r.boxes  # Boxes object for bbox outputs
    masks = r.masks  # Masks object for segment masks outputs
    probs = r.probs  # Class probabilities for classification outputs
    dir = r.save_dir
    print(boxes,masks,probs)
    print(dir)

**핵심**: YOLO 모델을 웹캠과 연결하면 실시간 객체 탐지를 구현할 수 있다. OpenCV와 supervision 라이브러리로 결과를 시각화한다.

---

## 7. 통합 정리

**복습 질문**

1. Bounding Box와 IOU는 각각 무엇을 의미하는가?
2. NMS가 없으면 탐지 결과가 어떻게 달라지는가?
3. 1-stage detector와 2-stage detector의 차이는 무엇인가?
4. YOLO가 실시간 탐지에 적합한 이유는 무엇인가?
5. mAP 점수가 높다고 항상 좋은 모델인가?

**심화 과제**

- 다른 데이터셋(COCO 부분집합, 자체 데이터)으로 YOLO 학습해 보기
- YOLOv8 대신 YOLOv11 등 최신 모델로 교체해 보기
- 탐지 결과를 영상 파일로 저장해 보기
